In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # H3 Feature Engineering from CARTO Marketplace
# MAGIC
# MAGIC **Simplified approach using CARTO pre-aggregated features**
# MAGIC
# MAGIC This notebook:
# MAGIC 1. Queries CARTO Marketplace spatial features table (demographics + POIs already aggregated at H3-8)
# MAGIC 2. Filters to Massachusetts H3 cells only
# MAGIC 3. Calculates total POI count from CARTO's 8 POI categories
# MAGIC 4. Adds distance to nearest LCE store
# MAGIC
# MAGIC **Data Source:** `carto_spatial_features_usa_h3_res_8.carto.derived_spatialfeatures_usa_h3res8_v1_yearly_v3`
# MAGIC
# MAGIC **CARTO POI Categories:** retail, education, financial, food_drink, healthcare, leisure, tourism, transportation
# MAGIC
# MAGIC **Output:** `h3_features_carto` table in **gold** schema

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import yaml

# Notebook parameters
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("silver_schema", "")
dbutils.widgets.text("gold_schema", "")
dbutils.widgets.text("state_fips", "")
dbutils.widgets.text("config_path", "")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")
state_fips = dbutils.widgets.get("state_fips")
config_path = dbutils.widgets.get("config_path")

assert catalog and bronze_schema and silver_schema and gold_schema and state_fips and config_path, \
    "Missing required parameters"

# Load configuration
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

H3_RESOLUTION = config['h3_grid']['resolution']

print(f"Catalog: {catalog}")
print(f"Target H3 Resolution: {H3_RESOLUTION}")
print(f"State FIPS: {state_fips}")

In [ ]:
# MAGIC %md
# MAGIC ## Step 1: Load Massachusetts Boundary

In [ ]:
# Load state boundary (Massachusetts)
state_df = spark.table(f"{catalog}.{bronze_schema}.census_states") \
    .filter(F.col("state_fips") == state_fips)

print(f"Loaded state boundary for FIPS: {state_fips}")
display(state_df.select("name", "state_abbr", "state_fips"))

In [ ]:
# MAGIC %md
# MAGIC ## Step 2: Generate H3 Grid for Massachusetts

In [ ]:
# Generate H3 cells covering Massachusetts using hierarchical approach
# Step 1: Coarse cover at resolution 5 (memory efficient)
coarse_cells_df = state_df.select(
    F.explode(F.expr("h3_coverash3string(ST_AsBinary(geometry), 5)")).alias("coarse_h3")
)

# Step 2: Explode to target resolution 8
h3_cells_df = coarse_cells_df.select(
    F.explode(F.expr(f"h3_tochildren(coarse_h3, {H3_RESOLUTION})")).alias("h3_cell_id")
)

# Step 3: Precise filtering using Point-in-Polygon
h3_cells_ma = h3_cells_df.join(
    F.broadcast(state_df),
    F.expr("ST_Contains(geometry, ST_GeomFromWKT(h3_centeraswkt(h3_cell_id), 4326))"),
    "inner"
).select("h3_cell_id").distinct()

ma_h3_count = h3_cells_ma.count()
print(f"Generated {ma_h3_count:,} H3 cells covering Massachusetts at resolution {H3_RESOLUTION}")
display(h3_cells_ma.limit(5))

In [ ]:
# MAGIC %md
# MAGIC ## Step 3: Load CARTO Marketplace Features
# MAGIC
# MAGIC CARTO table already contains demographics, income, education, AND POI counts aggregated at H3-8 resolution.

In [ ]:
# Load CARTO spatial features (entire USA at H3-8)
carto_table = "carto_spatial_features_usa_h3_res_8.carto.derived_spatialfeatures_usa_h3res8_v1_yearly_v3"

print(f"Loading CARTO features from: {carto_table}")
carto_features = spark.table(carto_table)

print(f"CARTO table schema ({len(carto_features.columns)} columns):")
print(carto_features.columns[:30])  # Show first 30 columns

In [ ]:
# Filter CARTO features to Massachusetts H3 cells only
carto_ma = carto_features.join(
    h3_cells_ma,
    carto_features["h3"] == h3_cells_ma["h3_cell_id"],
    "inner"
).drop(h3_cells_ma["h3_cell_id"]) \
 .withColumnRenamed("h3", "h3_cell_id")

ma_carto_count = carto_ma.count()
print(f"Filtered to {ma_carto_count:,} CARTO features in Massachusetts")
display(carto_ma.limit(5))

In [ ]:
# MAGIC %md
# MAGIC ## Step 4: Calculate Total POI Count from CARTO
# MAGIC
# MAGIC CARTO provides 8 pre-aggregated POI categories: retail, education, financial, food_drink, healthcare, leisure, tourism, transportation

In [ ]:
# Add total POI count from CARTO's pre-aggregated columns
carto_ma_with_pois = carto_ma.withColumn(
    "total_poi_count",
    (
        F.coalesce(F.col("retail"), F.lit(0)) +
        F.coalesce(F.col("education"), F.lit(0)) +
        F.coalesce(F.col("financial"), F.lit(0)) +
        F.coalesce(F.col("food_drink"), F.lit(0)) +
        F.coalesce(F.col("healthcare"), F.lit(0)) +
        F.coalesce(F.col("leisure"), F.lit(0)) +
        F.coalesce(F.col("tourism"), F.lit(0)) +
        F.coalesce(F.col("transportation"), F.lit(0))
    )
)

print(f"\nCARTO POI features summary:")
print(f"  - retail: {carto_ma_with_pois.agg(F.sum('retail')).collect()[0][0]:,}")
print(f"  - education: {carto_ma_with_pois.agg(F.sum('education')).collect()[0][0]:,}")
print(f"  - financial: {carto_ma_with_pois.agg(F.sum('financial')).collect()[0][0]:,}")
print(f"  - food_drink: {carto_ma_with_pois.agg(F.sum('food_drink')).collect()[0][0]:,}")
print(f"  - healthcare: {carto_ma_with_pois.agg(F.sum('healthcare')).collect()[0][0]:,}")
print(f"  - leisure: {carto_ma_with_pois.agg(F.sum('leisure')).collect()[0][0]:,}")
print(f"  - tourism: {carto_ma_with_pois.agg(F.sum('tourism')).collect()[0][0]:,}")
print(f"  - transportation: {carto_ma_with_pois.agg(F.sum('transportation')).collect()[0][0]:,}")
print(f"  - TOTAL: {carto_ma_with_pois.agg(F.sum('total_poi_count')).collect()[0][0]:,}")

display(carto_ma_with_pois.select(
    "h3_cell_id",
    "retail",
    "food_drink",
    "leisure",
    "total_poi_count"
).orderBy(F.desc("total_poi_count")).limit(10))

In [ ]:
# MAGIC %md
# MAGIC ## Step 5: Add Distance to Nearest LCE Store

In [ ]:
# Calculate H3 cell centers for distance calculations
h3_centers = h3_cells_ma.select(
    F.col("h3_cell_id"),
    F.expr("ST_GeomFromWKT(h3_centeraswkt(h3_cell_id), 4326)").alias("h3_center_point")
)

# Load LCE locations
lce_df = spark.table(f"{catalog}.{bronze_schema}.lce_locations_mass") \
    .select(
        F.expr("ST_Point(longitude, latitude, 4326)").alias("lce_point")
    )

lce_count = lce_df.count()
print(f"Loaded {lce_count} LCE store locations")

# Calculate distances (broadcast small LCE table)
h3_lce_distances = h3_centers.crossJoin(F.broadcast(lce_df)) \
    .withColumn(
        "distance_miles",
        F.expr("ST_Distance(h3_center_point, lce_point) / 1609.34")
    )

# Get minimum distance per H3 cell
lce_distance_features = h3_lce_distances.groupBy("h3_cell_id") \
    .agg(F.min("distance_miles").alias("distance_to_nearest_lce_miles"))

print(f"Calculated LCE distances for {lce_distance_features.count():,} H3 cells")
display(lce_distance_features.orderBy("distance_to_nearest_lce_miles").limit(10))

In [ ]:
# MAGIC %md
# MAGIC ## Step 6: Join All Features

In [ ]:
# Join CARTO features (with POIs already included) with LCE distances
h3_features_final = carto_ma_with_pois \
    .join(lce_distance_features, "h3_cell_id", "left") \
    .withColumn("processing_timestamp", F.current_timestamp())

# Fill null distances with high value (cells far from any store)
null_distance_value = config.get('distance', {}).get('null_value', 999)
h3_features_final = h3_features_final.fillna(null_distance_value, subset=["distance_to_nearest_lce_miles"])

# Fill nulls for CARTO POI columns
carto_poi_cols = ["retail", "education", "financial", "food_drink", "healthcare", "leisure", "tourism", "transportation", "total_poi_count"]
h3_features_final = h3_features_final.fillna(0, subset=carto_poi_cols)

final_count = h3_features_final.count()
print(f"\n✓ Final H3 features: {final_count:,} cells")
print(f"  - CARTO demographic features: ✓")
print(f"  - CARTO POI counts (8 categories): ✓")
print(f"  - LCE store distances: ✓")

display(h3_features_final.select(
    "h3_cell_id",
    "population",
    "food_drink",
    "retail",
    "total_poi_count",
    "distance_to_nearest_lce_miles"
).orderBy(F.desc("total_poi_count")).limit(10))

In [ ]:
# MAGIC %md
# MAGIC ## Step 7: Write to Gold Table

In [ ]:
# Write to gold schema (CARTO-based H3 features for LCE)
output_table = f"{catalog}.{gold_schema}.h3_features_carto"

h3_features_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(output_table)

print(f"\n✅ SUCCESS: Wrote {final_count:,} H3 features to {output_table}")
print(f"\nThis table combines:")
print(f"  1. CARTO marketplace demographics (pre-aggregated at H3-8)")
print(f"  2. CARTO POI counts - 8 categories: retail, education, financial, food_drink, healthcare, leisure, tourism, transportation")
print(f"  3. Distance to nearest Little Caesars store")

In [ ]:
# MAGIC %md
# MAGIC ## Step 8: Summary Statistics

In [ ]:
# Show summary statistics
display(spark.sql(f"""
    SELECT
        COUNT(*) as total_h3_cells,
        ROUND(AVG(population), 0) as avg_population,
        ROUND(AVG(retail), 0) as avg_retail_pois,
        ROUND(AVG(food_drink), 0) as avg_food_drink_pois,
        ROUND(AVG(total_poi_count), 0) as avg_total_pois,
        ROUND(AVG(distance_to_nearest_lce_miles), 2) as avg_distance_to_lce,
        ROUND(MIN(distance_to_nearest_lce_miles), 2) as min_distance_to_lce,
        ROUND(MAX(distance_to_nearest_lce_miles), 2) as max_distance_to_lce
    FROM {output_table}
"""))